<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/inspect-adapters.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
# !pip -qqq install peft trl

In [ ]:
# !unzip lora-adapter_1750_1760.zip -d .

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
from collections import defaultdict
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")

In [ ]:
def compare_peft_lora_adapters(
    base_model_name_or_path: str,
    adapter_path_1: str,
    adapter_path_2: str,
    device: str = "cpu",
    atol: float = 1e-6,
):
    """
    Compare two PEFT LoRA adapters trained on the same base model.

    Args:
        base_model_name_or_path: HF model id or local path
        adapter_path_1: path to first adapter
        adapter_path_2: path to second adapter
        device: cpu or cuda
        atol: tolerance for equality check

    Returns:
        dict with per-tensor difference statistics
    """

    # Load base model once
    base_model_1 = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
    ).to(device)

    base_model_2 = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
    ).to(device)

    # Load adapters separately
    model1 = PeftModel.from_pretrained(base_model_1, adapter_path_1).to(device)
    model2 = PeftModel.from_pretrained(base_model_2, adapter_path_2).to(device)

    # Extract only LoRA weights
    sd1 = {
        k: v for k, v in model1.state_dict().items()
        if "lora_" in k
    }
    sd2 = {
        k: v for k, v in model2.state_dict().items()
        if "lora_" in k
    }

    report = {}
    changed = []

    for key in sd1.keys():
        if key not in sd2:
            print(f"[Missing in adapter2] {key}")
            continue

        w1 = sd1[key]
        w2 = sd2[key]

        diff = w1 - w2

        l2 = torch.norm(diff).item()
        max_abs = diff.abs().max().item()
        rel = l2 / (torch.norm(w1).item() + 1e-12)

        cos_dist = 1.0 -  torch.nn.functional.cosine_similarity(
            w1.flatten(), w2.flatten(), dim=0
                ).item()

        equal = torch.allclose(w1, w2, atol=atol)

        report[key] = {
            "equal": equal,
            "l2_diff": l2,
            "max_abs_diff": max_abs,
            "relative_l2": rel,
            'cosine': cos_dist,
        }

        if not equal:
            changed.append(key)

    print(f"\nTotal LoRA tensors: {len(sd1)}")
    print(f"Changed tensors: {len(changed)}")

    return report


def plot_lora_difference_heatmap(report, stat='cosine'):
    """
    Creates a heatmap of LoRA L2 differences aggregated by:
    Transformer layer × projection module (q_proj, k_proj, etc.)

    Args:
        report: output dict from compare_peft_lora_adapters()
    """

    # Aggregate differences
    layer_module_diff = defaultdict(lambda: defaultdict(float))
    modules = set()
    layers = set()

    for key, stats in report.items():
        if stats["l2_diff"] == 0:
            continue

        parts = key.split(".")

        # Extract layer number
        if "layers" in parts:
            layer_idx = int(parts[parts.index("layers") + 1])
        else:
            continue

        # Extract projection/module name
        # e.g. q_proj, k_proj, v_proj, o_proj, gate_proj, etc.
        module_name = None
        for part in parts:
            if part.endswith("_proj"):
                module_name = part
                break

        if module_name is None:
            continue

        layer_module_diff[layer_idx][module_name] += stats[stat]
        modules.add(module_name)
        layers.add(layer_idx)

    layers = sorted(layers)
    modules = sorted(modules)

    # Build matrix
    heatmap = np.zeros((len(layers), len(modules)))

    for i, layer in enumerate(layers):
        for j, module in enumerate(modules):
            heatmap[i, j] = layer_module_diff[layer].get(module, 0.0)

    # Single plot only
    plt.figure()
    plt.imshow(heatmap)
    plt.xticks(range(len(modules)), modules, rotation=45)
    plt.yticks(range(len(layers)), layers)
    plt.xlabel("Projection Module")
    plt.ylabel("Transformer Layer")
    plt.title(f"LoRA {stat.upper()} Heatmap")
    plt.colorbar()
    plt.tight_layout()
    plt.show()



In [ ]:
base_model_name_or_path = 'meta-llama/Meta-Llama-3-8B'
adapter_path_1 = './lora-adapter_1750_1760/checkpoint-1000'
adapter_path_2 = './lora-adapter_1750_1760/checkpoint-5393'

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:

report = compare_peft_lora_adapters(base_model_name_or_path,adapter_path_1,adapter_path_2)
plot_lora_difference_heatmap(report, 'cosine')